# 03 章 / 第 2 节 · 中文 benchmark:CMMLU + C-Eval

## 学习目标
- 跑通 CMMLU + C-Eval 两个中文 benchmark baseline
- 理解二者差异(领域分布、few-shot 设置、是否 dev 集校准)
- 关注 STEM / 人文 / 社科分项 —— 中文模型常常 STEM 弱、人文强
- 为后续 SFT/对齐章节留中文 baseline 数字

## 对应八股
`liguodongiot/llm-action`:`llm-interview/llm-eval.md`(中文 benchmark 节)

## ⚠️ 运行要求
Linux + CUDA(vLLM)或 transformers backend;CMMLU 67 学科约 11.5k 题,vLLM ~30 min。


## 1. CMMLU vs C-Eval 速览

| 维度 | CMMLU | C-Eval |
|---|---|---|
| 题数 | ~11.5k 多选 | ~13.9k 多选 |
| 学科 | 67(STEM/人文/社科/中国特色) | 52(类似分布) |
| Few-shot | 0/5-shot 任选 | 推荐 5-shot |
| Dev/Test 区分 | 有 dev 校准 | 有 dev 校准 |
| 一句话 | 港中文出品,更偏知识题 | 港科出品,偏推理题 |

两个都用,看 **一致性**:CMMLU 高 C-Eval 低 → 模型擅长背书不擅长推理。

## 2. 环境检查


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from scripts.env_check import check
check(extras=("lm_eval",))


## 3. 加载模型 + 跑 CMMLU


In [ ]:
import lm_eval
from lm_eval.models.huggingface import HFLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

try:
    from lm_eval.models.vllm_causallms import VLLM
    model = VLLM(pretrained=MODEL_NAME, dtype="bfloat16", max_model_len=2048,
                 gpu_memory_utilization=0.8, batch_size="auto")
except Exception:
    model = HFLM(pretrained=MODEL_NAME, dtype="bfloat16", batch_size=4)

# CMMLU 任务名约定:cmmlu_subject 或 cmmlu(总)
CMMLU_SUBJECTS = [
    "cmmlu_chinese_history",       # 人文
    "cmmlu_high_school_mathematics", # STEM
    "cmmlu_high_school_geography",  # 社科
    "cmmlu_marketing",              # 商业
]

cmmlu_res = lm_eval.simple_evaluate(
    model = model,
    tasks = CMMLU_SUBJECTS,
    num_fewshot = 0,
    limit = 100,                    # 每科 100 题
    batch_size = "auto",
)
print(f"完成 {len(cmmlu_res['results'])} 个 CMMLU 子集")


## 4. 跑 C-Eval


In [ ]:
CEVAL_SUBJECTS = [
    "ceval-valid_chinese_language_and_literature",
    "ceval-valid_high_school_mathematics",
    "ceval-valid_marxism",
    "ceval-valid_computer_network",
]
# 注意:任务名前缀是 ceval-valid_ 或 ceval-test_;开发者用 valid 设了 dev 集校准

ceval_res = lm_eval.simple_evaluate(
    model = model,
    tasks = CEVAL_SUBJECTS,
    num_fewshot = 5,                # C-Eval 推荐 5-shot
    limit = 100,
    batch_size = "auto",
)
print(f"完成 {len(ceval_res['results'])} 个 C-Eval 子集")


## 5. 中文 baseline 表


In [ ]:
import pandas as pd
import json

rows = []
for task, scores in {**cmmlu_res["results"], **ceval_res["results"]}.items():
    rows.append({
        "task": task,
        "acc":      f"{scores.get('acc,none', float('nan'))*100:.1f}%",
        "acc_norm": f"{scores.get('acc_norm,none', float('nan'))*100:.1f}%",
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))

# 存到 CSV
pathlib.Path("../checkpoints/03_eval").mkdir(parents=True, exist_ok=True)
df.to_csv("../checkpoints/03_eval/baseline_zh.csv", index=False)
with open("../checkpoints/03_eval/baseline_zh.json", "w") as f:
    json.dump({**cmmlu_res["results"], **ceval_res["results"]}, f,
              ensure_ascii=False, indent=2, default=str)
print("\n已存到 ../checkpoints/03_eval/baseline_zh.{csv,json}")


## 6. 踩坑记录

- **CMMLU / C-Eval 任务名前缀**:lm-eval 加 `cmmlu_` / `ceval-valid_` 或 `ceval-test_`,版本不同可能有变,失败先 `lm_eval --tasks list | grep cmmlu` 确认
- **C-Eval test 集分数不在 leaderboard 上**:test 答案不公开,要提交 leaderboard 才能看,本地只能跑 valid;valid ≈ test 的代理
- **`num_fewshot=5` 的样本占 context**:max_model_len=2048 时 5-shot + 题目可能超 1500 token,Qwen-1.5B 的 max 是 32k 没问题,但小模型(SmolLM2-360M)要降到 3-shot
- **STEM 题 acc 通常比人文低 20-30pt**:不是模型坏,是 1.5B 这个尺寸数学推理天花板就低,看相对差异不要看绝对

## 7. 延伸阅读

- [CMMLU 仓库](https://github.com/haonan-li/CMMLU)
- [C-Eval 仓库](https://github.com/hkust-nlp/ceval)
- 论文:CMMLU `/paper fetch 2306.09212`、C-Eval `/paper fetch 2305.08322`
- [[Slipbox/chinese-llm-benchmarks]]
